# e-Procurement Risk Analytics - Data Conversion & ML Training

This notebook demonstrates:
1. Converting JSON procurement data to CSV
2. Exploratory Data Analysis (EDA)
3. Training Isolation Forest for anomaly detection
4. Evaluating and visualizing results

## Requirements
```bash
pip install pandas numpy scikit-learn matplotlib seaborn jupyter
```

## 1. Setup and Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

## 2. Load and Convert JSON to CSV

In [ ]:
def flatten_notice_item(item):
    """Flatten nested JSON structure for CSV export"""
    return {
        # Basic identifiers
        'caNoticeId': item.get('caNoticeId', ''),
        'noticeId': item.get('noticeId', ''),
        'procedureId': item.get('procedureId', ''),
        'noticeNo': item.get('noticeNo', ''),
        
        # Notice state
        'sysNoticeTypeId': item.get('sysNoticeTypeId', ''),
        'noticeStateId': item.get('sysNoticeState', {}).get('id', ''),
        'noticeStateText': item.get('sysNoticeState', {}).get('text', ''),
        
        # Procedure state
        'procedureStateId': item.get('sysProcedureState', {}).get('id', ''),
        'procedureStateText': item.get('sysProcedureState', {}).get('text', ''),
        
        # Contracting authority
        'contractingAuthority': item.get('contractingAuthorityNameAndFN', ''),
        'contractTitle': item.get('contractTitle', ''),
        
        # Contract types
        'acquisitionContractTypeId': item.get('sysAcquisitionContractType', {}).get('id', ''),
        'acquisitionContractTypeText': item.get('sysAcquisitionContractType', {}).get('text', ''),
        'procedureTypeId': item.get('sysProcedureType', {}).get('id', ''),
        'procedureTypeText': item.get('sysProcedureType', {}).get('text', ''),
        'contractAssigmentTypeId': item.get('sysContractAssigmentType', {}).get('id', ''),
        'contractAssigmentTypeText': item.get('sysContractAssigmentType', {}).get('text', ''),
        
        # CPV and values
        'cpvCodeAndName': item.get('cpvCodeAndName', ''),
        'ronContractValue': item.get('ronContractValue', 0),
        'estimatedValueExport': item.get('estimatedValueExport', ''),
        'highestOfferValue': item.get('highestOfferValue', 0),
        'lowestOfferValue': item.get('lowestOfferValue', 0),
        
        # Dates
        'noticeStateDate': item.get('noticeStateDate', ''),
        'minTenderReceiptDeadline': item.get('minTenderReceiptDeadline', ''),
        'maxTenderReceiptDeadline': item.get('maxTenderReceiptDeadline', ''),
        'tenderReceiptDeadlineExport': item.get('tenderReceiptDeadlineExport', ''),
        
        # Flags
        'isOnline': item.get('isOnline', False),
        'hasSubsequentContracts': item.get('hasSubsequentContracts', False),
        'isUtility': item.get('isUtility', False),
        'isPPP': item.get('isPPP', False),
        
        # Other fields
        'errataNo': item.get('errataNo', 0),
        'versionNo': item.get('versionNo', ''),
        'sysNoticeVersionId': item.get('sysNoticeVersionId', ''),
        'currencyCode': item.get('currencyCode', 'RON'),
    }

# Load JSON data
json_file = '../public/data.json'
print(f"Loading data from {json_file}...")

with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total records in JSON: {data.get('total', 'unknown')}")
print(f"Items to process: {len(data['items'])}")

In [ ]:
# Convert to DataFrame
print("Converting to DataFrame...")
rows = [flatten_notice_item(item) for item in data['items']]
df = pd.DataFrame(rows)

print(f"✓ DataFrame created with {len(df)} rows and {len(df.columns)} columns")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst few columns: {list(df.columns[:5])}")

# Display first few rows
df.head()

In [ ]:
# Save to CSV
output_dir = Path('../output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y-%m-%d')
csv_file = output_dir / f'procurement-training-data-{timestamp}.csv'

df.to_csv(csv_file, index=False)
file_size_mb = csv_file.stat().st_size / (1024 * 1024)

print(f"✓ CSV saved to: {csv_file}")
print(f"✓ File size: {file_size_mb:.2f} MB")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data types
print("\nData types:")
print(df.dtypes.value_counts())

# Missing values
print("\nMissing values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].sort_values(ascending=False))
else:
    print("No missing values!")

In [ ]:
# Contract value statistics
print("\n" + "=" * 60)
print("CONTRACT VALUE STATISTICS")
print("=" * 60)
print(df['ronContractValue'].describe())

# Distribution of contract values
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df['ronContractValue'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Contract Value (RON)')
plt.ylabel('Frequency')
plt.title('Distribution of Contract Values')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Log scale for better visualization
df_nonzero = df[df['ronContractValue'] > 0]
plt.hist(np.log10(df_nonzero['ronContractValue']), bins=50, edgecolor='black', alpha=0.7, color='orange')
plt.xlabel('Log10(Contract Value)')
plt.ylabel('Frequency')
plt.title('Distribution of Contract Values (Log Scale)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Procedure types distribution
print("\n" + "=" * 60)
print("PROCEDURE TYPES")
print("=" * 60)
procedure_counts = df['procedureTypeText'].value_counts()
print(procedure_counts)

# Visualize
plt.figure(figsize=(12, 6))
procedure_counts.head(10).plot(kind='barh', color='steelblue')
plt.xlabel('Count')
plt.title('Top 10 Procedure Types')
plt.tight_layout()
plt.show()

In [ ]:
# Acquisition types
print("\n" + "=" * 60)
print("ACQUISITION TYPES")
print("=" * 60)
acquisition_counts = df['acquisitionContractTypeText'].value_counts()
print(acquisition_counts)

# Pie chart
plt.figure(figsize=(10, 6))
plt.pie(acquisition_counts.values, labels=acquisition_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Acquisition Types')
plt.axis('equal')
plt.show()

## 4. Feature Engineering for ML

In [ ]:
# Select features for anomaly detection
feature_columns = [
    'ronContractValue',
    'highestOfferValue',
    'lowestOfferValue',
    'procedureTypeId',
    'acquisitionContractTypeId',
    'errataNo',
    'noticeStateId',
    'procedureStateId'
]

print("Selected features for ML:")
for col in feature_columns:
    print(f"  - {col}")

# Create feature matrix
X = df[feature_columns].fillna(0).copy()

# Add binary flags as features
X['isOnline'] = df['isOnline'].astype(int)
X['isPPP'] = df['isPPP'].astype(int)
X['isUtility'] = df['isUtility'].astype(int)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")

# Display feature statistics
X.describe()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation = X.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Train Isolation Forest Model

In [ ]:
# Normalize features
print("Normalizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✓ Features normalized")
print(f"Mean: {X_scaled.mean(axis=0)[:3]}...")
print(f"Std: {X_scaled.std(axis=0)[:3]}...")

In [ ]:
# Train Isolation Forest
print("\n" + "=" * 60)
print("TRAINING ISOLATION FOREST")
print("=" * 60)

contamination = 0.1  # Expected anomaly rate (10%)
n_estimators = 100   # Number of trees

print(f"Parameters:")
print(f"  - Trees: {n_estimators}")
print(f"  - Contamination: {contamination*100}%")
print(f"  - Random state: 42")

model = IsolationForest(
    n_estimators=n_estimators,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nTraining model...")
predictions = model.fit_predict(X_scaled)
scores = model.score_samples(X_scaled)

print("✓ Model trained successfully!")

## 6. Analyze Results

In [ ]:
# Add predictions to dataframe
df['anomaly_score'] = scores
df['is_anomaly'] = predictions == -1
df['prediction'] = predictions

# Count results
n_anomalies = (predictions == -1).sum()
n_normal = (predictions == 1).sum()

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"Total records: {len(df)}")
print(f"Normal points: {n_normal} ({n_normal/len(df)*100:.1f}%)")
print(f"Anomalies detected: {n_anomalies} ({n_anomalies/len(df)*100:.1f}%)")
print(f"\nAnomaly score range: [{scores.min():.4f}, {scores.max():.4f}]")

In [ ]:
# Visualize anomaly scores
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(scores, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Anomaly Score')
plt.ylabel('Frequency')
plt.title('Distribution of Anomaly Scores')
plt.axvline(x=scores[predictions == -1].max(), color='red', linestyle='--', label='Anomaly threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
colors = ['red' if p == -1 else 'blue' for p in predictions]
plt.scatter(range(len(scores)), sorted(scores), c=sorted(colors), alpha=0.5, s=10)
plt.xlabel('Sample Index (sorted)')
plt.ylabel('Anomaly Score')
plt.title('Anomaly Scores (Sorted)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 anomalies
print("\n" + "=" * 60)
print("TOP 20 ANOMALIES (Most Suspicious)")
print("=" * 60)

anomalies = df[df['is_anomaly']].copy()
anomalies = anomalies.sort_values('anomaly_score')

for idx, (_, row) in enumerate(anomalies.head(20).iterrows(), 1):
    print(f"\n{idx}. Notice {row['noticeNo']}")
    print(f"   Authority: {row['contractingAuthority'][:50]}...")
    print(f"   Title: {row['contractTitle'][:60]}...")
    print(f"   Value: {row['ronContractValue']:,.0f} RON")
    print(f"   Procedure: {row['procedureTypeText']}")
    print(f"   Anomaly Score: {row['anomaly_score']:.6f}")

In [ ]:
# Compare feature distributions between normal and anomalies
print("\n" + "=" * 60)
print("FEATURE COMPARISON: Normal vs Anomalies")
print("=" * 60)

comparison_features = ['ronContractValue', 'highestOfferValue', 'lowestOfferValue', 'errataNo']

for col in comparison_features:
    normal_mean = df[~df['is_anomaly']][col].mean()
    anomaly_mean = df[df['is_anomaly']][col].mean()
    
    if normal_mean != 0:
        diff_pct = ((anomaly_mean - normal_mean) / normal_mean) * 100
        print(f"\n{col}:")
        print(f"  Normal avg:   {normal_mean:>15,.2f}")
        print(f"  Anomaly avg:  {anomaly_mean:>15,.2f}")
        print(f"  Difference:   {diff_pct:>15.1f}%")

In [ ]:
# Visualize feature comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(comparison_features):
    ax = axes[idx]
    
    df[~df['is_anomaly']][col].hist(bins=30, alpha=0.5, label='Normal', ax=ax, color='blue')
    df[df['is_anomaly']][col].hist(bins=30, alpha=0.5, label='Anomaly', ax=ax, color='red')
    
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution: {col}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. PCA Visualization

In [ ]:
# Reduce to 2D for visualization
print("Performing PCA for visualization...")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"✓ PCA complete")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Plot
plt.figure(figsize=(12, 8))
colors = ['red' if p == -1 else 'blue' for p in predictions]
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.5, s=30)
plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Procurement Data in 2D (PCA)\nRed = Anomalies, Blue = Normal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
# Save anomalies to CSV
anomalies_file = output_dir / 'detected-anomalies.csv'
anomalies.to_csv(anomalies_file, index=False)
print(f"✓ Anomalies saved to: {anomalies_file}")

# Save full dataset with predictions
full_results_file = output_dir / f'procurement-with-predictions-{timestamp}.csv'
df.to_csv(full_results_file, index=False)
print(f"✓ Full results saved to: {full_results_file}")

# Save summary statistics
summary_file = output_dir / f'analysis-summary-{timestamp}.txt'
with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("PROCUREMENT RISK ANALYTICS - ANALYSIS SUMMARY\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"Dataset: {len(df)} procurement notices\n")
    f.write(f"Features: {len(X.columns)}\n\n")
    f.write(f"Model: Isolation Forest\n")
    f.write(f"  - Trees: {n_estimators}\n")
    f.write(f"  - Contamination: {contamination*100}%\n\n")
    f.write(f"Results:\n")
    f.write(f"  - Normal: {n_normal} ({n_normal/len(df)*100:.1f}%)\n")
    f.write(f"  - Anomalies: {n_anomalies} ({n_anomalies/len(df)*100:.1f}%)\n\n")
    f.write(f"Output files:\n")
    f.write(f"  - {anomalies_file}\n")
    f.write(f"  - {full_results_file}\n")

print(f"✓ Summary saved to: {summary_file}")

print("\n" + "=" * 60)
print("✓ ANALYSIS COMPLETE")
print("=" * 60)

## 9. Next Steps

### Recommendations:

1. **Review Detected Anomalies**
   - Check `output/detected-anomalies.csv`
   - Validate findings with domain experts
   - Identify patterns in flagged contracts

2. **Tune the Model**
   - Adjust `contamination` parameter based on validation
   - Try different feature combinations
   - Experiment with `n_estimators`

3. **Feature Engineering**
   - Add deadline urgency (days until deadline)
   - Calculate authority history metrics
   - Include temporal features (month, day of week)
   - Add CPV category indicators

4. **Try Other Models**
   - Local Outlier Factor (LOF)
   - One-Class SVM
   - Autoencoders (deep learning)

5. **Integration**
   - Combine ML scores with rule-based risk scores
   - Create automated alerts for high-risk contracts
   - Build a dashboard for real-time monitoring

### Additional Analysis Ideas:

- Time series analysis of contract trends
- Network analysis of authority-vendor relationships
- Clustering to identify contract groups
- Supervised learning if labeled fraud data becomes available